# Comparison Notebook

This notebook regenerates the retained manuscript and supplementary comparison assets from the same dataset and matrix sources used by main.ipynb.

Output policy:
- Export only CSV and PDF assets directly into docs/DCHE-D-26-00020.
- Do not persist analysis/training artifacts to results/.

In [ ]:
import hashlib
import json
import pickle
import re
import subprocess
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Mapping
import tomllib

import numpy as np
import pandas as pd
from openpyxl import load_workbook
from sklearn.model_selection import train_test_split


ASM2D_TSN_MODEL_NAME = "asm2d_tsn_simulation"
ASM2D_TSN_WORKBOOK_PATH_KEY = "asm2d_tsn_reference_workbook"
ASM2D_TSN_COMPOSITION_CACHE_PATTERN_KEY = "asm2d_tsn_composition_cache_pattern"
ASM2D_TSN_COMPOSITION_CACHE_METADATA_PATTERN_KEY = "asm2d_tsn_composition_cache_metadata_pattern"
STOICHIOMETRIC_SHEET_NAME = "stoichiometric_matrix"
COMPOSITION_SHEET_NAME = "composition_matrix"
PARAMETER_SHEET_NAME = "parameter_table"

_EXCEL_REFERENCE_PATTERN = re.compile(
    r"(?<![A-Za-z0-9_])(?:(?:'(?P<sheet_quoted>[^']+)')|(?P<sheet_unquoted>[A-Za-z_][A-Za-z0-9_]*))?!?\$?(?P<column>[A-Za-z]{1,3})\$?(?P<row>\d+)"
)

STOICHIOMETRIC_COEFFICIENTS = [
    {"coefficients": {"S_F": "1-{f_SI}", "S_I": "{f_SI}", "X_S": "-1"}},
    {"coefficients": {"S_F": "1-{f_SI}", "S_I": "{f_SI}", "X_S": "-1"}},
    {"coefficients": {"S_F": "1-{f_SI}", "S_I": "{f_SI}", "X_S": "-1"}},
    {"coefficients": {"S_F": "1-{f_SI}", "S_I": "{f_SI}", "X_S": "-1"}},
    {"coefficients": {"S_A": "-1/{Y_H}", "S_O": "1-1/{Y_H}", "X_H": "1"}},
    {"coefficients": {"S_A": "-1/{Y_H}", "S_O": "1-1/{Y_H}", "X_H": "1"}},
    {
        "coefficients": {
            "S_F": "-1/{Y_H}",
            "S_NO2": "(1-{Y_H})/((8/7)*{Y_H})",
            "S_NO3": "-((1-{Y_H})/((8/7)*{Y_H}))",
            "X_H": "1",
        }
    },
    {
        "coefficients": {
            "S_F": "-1/{Y_H}",
            "S_N2": "(1-{Y_H})/(1.72*{Y_H})",
            "S_NO2": "-((1-{Y_H})/(1.72*{Y_H}))",
            "X_H": "1",
        }
    },
    {
        "coefficients": {
            "S_A": "-1/{Y_H}",
            "S_NO2": "(1-{Y_H})/((8/7)*{Y_H})",
            "S_NO3": "-((1-{Y_H})/((8/7)*{Y_H}))",
            "X_H": "1",
        }
    },
    {
        "coefficients": {
            "S_A": "-1/{Y_H}",
            "S_N2": "(1-{Y_H})/(1.72*{Y_H})",
            "S_NO2": "-((1-{Y_H})/(1.72*{Y_H}))",
            "X_H": "1",
        }
    },
    {"coefficients": {"S_A": "1", "S_F": "-1"}},
    {"coefficients": {"X_I": "{f_XI}", "X_S": "1-{f_XI}", "X_H": "-1"}},
    {"coefficients": {"S_A": "-1", "X_PP": "-{Y_PO4}", "X_PHA": "1"}},
    {"coefficients": {"S_O": "-{Y_PHA}", "X_PP": "1", "X_PHA": "-{Y_PHA}"}},
    {
        "coefficients": {
            "S_NO2": "{Y_PHA}/(8/7)",
            "S_NO3": "-({Y_PHA}/(8/7))",
            "X_PP": "1",
            "X_PHA": "-{Y_PHA}",
        }
    },
    {
        "coefficients": {
            "S_N2": "{Y_PHA}/1.72",
            "S_NO2": "-({Y_PHA}/1.72)",
            "X_PP": "1",
            "X_PHA": "-{Y_PHA}",
        }
    },
    {"coefficients": {"S_O": "1-1/{Y_PAO}", "X_PAO": "1", "X_PHA": "-1/{Y_PAO}"}},
    {
        "coefficients": {
            "S_NO2": "(1-{Y_PAO})/((8/7)*{Y_PAO})",
            "S_NO3": "-((1-{Y_PAO})/((8/7)*{Y_PAO}))",
            "X_PAO": "1",
            "X_PHA": "-1/{Y_PAO}",
        }
    },
    {
        "coefficients": {
            "S_N2": "(1-{Y_PAO})/(1.72*{Y_PAO})",
            "S_NO2": "-((1-{Y_PAO})/(1.72*{Y_PAO}))",
            "X_PAO": "1",
            "X_PHA": "-1/{Y_PAO}",
        }
    },
    {"coefficients": {"X_I": "{f_XI}", "X_S": "1-{f_XI}", "X_PAO": "-1"}},
    {"coefficients": {"X_PP": "-1"}},
    {"coefficients": {"S_A": "1", "X_PHA": "-1"}},
    {"coefficients": {"S_NO2": "1/{Y_AOB}", "S_O": "-((3.43-{Y_AOB})/{Y_AOB})", "X_AOB": "1"}},
    {"coefficients": {"S_NO2": "-1/{Y_NOB}", "S_NO3": "1/{Y_NOB}", "S_O": "-((1.14-{Y_NOB})/{Y_NOB})", "X_NOB": "1"}},
    {"coefficients": {"X_I": "{f_XI}", "X_S": "1-{f_XI}", "X_AOB": "-1"}},
    {"coefficients": {"X_I": "{f_XI}", "X_S": "1-{f_XI}", "X_NOB": "-1"}},
    {"coefficients": {"S_PO4": "-1", "X_MeOH": "-3.45", "X_MeP": "4.87"}},
    {"coefficients": {"S_PO4": "1", "X_MeOH": "3.45", "X_MeP": "-4.87"}},
]

NITROGEN_CONTINUITY_TERMS = {
    "S_F": "{i_NSF}",
    "S_I": "{i_NSI}",
    "S_N2": "1",
    "S_NO2": "1",
    "S_NO3": "1",
    "X_I": "{i_NXI}",
    "X_S": "{i_NXS}",
    "X_H": "{i_NBM}",
    "X_PAO": "{i_NBM}",
    "X_AOB": "{i_NBM}",
    "X_NOB": "{i_NBM}",
}

PHOSPHORUS_CONTINUITY_TERMS = {
    "S_F": "{i_PSF}",
    "S_I": "{i_PSI}",
    "X_I": "{i_PXI}",
    "X_S": "{i_PXS}",
    "X_H": "{i_PBM}",
    "X_PAO": "{i_PBM}",
    "X_PP": "1",
    "X_AOB": "{i_PBM}",
    "X_NOB": "{i_PBM}",
    "X_MeP": "{i_PMeP}",
}


def find_repo_root(start_path: Path) -> Path:
    for candidate in (start_path, *start_path.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not locate pyproject.toml from the current working directory.")


NOTEBOOK_REPO_ROOT = find_repo_root(Path.cwd().resolve())


def load_json_file(file_path: str | Path) -> dict[str, Any]:
    return json.loads(Path(file_path).read_text(encoding="utf-8-sig"))


def save_json_file(file_path: str | Path, data: dict[str, Any], *, indent: int = 2) -> Path:
    path = Path(file_path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        json.dump(data, handle, indent=indent)
        handle.write("\n")
    return path


def load_pickle_file(file_path: str | Path) -> Any:
    with Path(file_path).open("rb") as handle:
        return pickle.load(handle)


def save_pickle_file(file_path: str | Path, data: Any) -> Path:
    path = Path(file_path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("wb") as handle:
        pickle.dump(data, handle)
    return path


def compute_file_sha256(file_path: str | Path, *, chunk_size: int = 1024 * 1024) -> str:
    hasher = hashlib.sha256()
    with Path(file_path).open("rb") as handle:
        while True:
            chunk = handle.read(int(chunk_size))
            if not chunk:
                break
            hasher.update(chunk)
    return hasher.hexdigest()


def get_repo_root(repo_root: str | Path | None = None) -> Path:
    if repo_root is None:
        return NOTEBOOK_REPO_ROOT
    return Path(repo_root).resolve()


def load_paths_config(repo_root: str | Path | None = None) -> dict[str, Any]:
    root = get_repo_root(repo_root)
    return load_json_file(root / "config" / "paths.json")


def load_params_config(repo_root: str | Path | None = None) -> dict[str, Any]:
    root = get_repo_root(repo_root)
    return load_json_file(root / "config" / "params.json")


def load_ml_orchestration_params(repo_root: str | Path | None = None) -> dict[str, Any]:
    params = load_params_config(repo_root)
    if "ml_orchestration" not in params:
        raise KeyError("Machine-learning orchestration parameters not found.")
    return params["ml_orchestration"]


def load_model_params(model_name: str, repo_root: str | Path | None = None) -> dict[str, Any]:
    params = load_params_config(repo_root)
    if model_name not in params:
        raise KeyError(f"Model parameters not found for '{model_name}'.")
    return params[model_name]


def load_asm2d_tsn_simulation_params(repo_root: str | Path | None = None) -> dict[str, Any]:
    return load_model_params(ASM2D_TSN_MODEL_NAME, repo_root)


def resolve_asm2d_tsn_workbook_path(
    repo_root: str | Path | None = None,
    *,
    paths_config: Mapping[str, Any] | None = None,
) -> Path:
    root = get_repo_root(repo_root)
    config = dict(paths_config) if paths_config is not None else load_paths_config(root)
    return root / Path(config[ASM2D_TSN_WORKBOOK_PATH_KEY])


def resolve_asm2d_tsn_composition_cache_paths(
    workbook_hash: str,
    repo_root: str | Path | None = None,
    *,
    paths_config: Mapping[str, Any] | None = None,
) -> tuple[Path, Path]:
    root = get_repo_root(repo_root)
    config = dict(paths_config) if paths_config is not None else load_paths_config(root)
    matrix_path = root / Path(
        config[ASM2D_TSN_COMPOSITION_CACHE_PATTERN_KEY].format(workbook_hash=str(workbook_hash))
    )
    metadata_path = root / Path(
        config[ASM2D_TSN_COMPOSITION_CACHE_METADATA_PATTERN_KEY].format(workbook_hash=str(workbook_hash))
    )
    return matrix_path, metadata_path


def _validate_unique_names(names: list[str], name_type: str) -> None:
    if not names:
        raise ValueError(f"{name_type} must not be empty.")
    duplicates = sorted({name for name in names if names.count(name) > 1})
    if duplicates:
        duplicate_display = ", ".join(duplicates)
        raise ValueError(f"asm2d_tsn_simulation {name_type} contains duplicates: {duplicate_display}")


def _validate_workbook_config(model_params: Mapping[str, Any]) -> dict[str, Any]:
    if "workbook" not in model_params:
        raise KeyError("asm2d_tsn_simulation must define a workbook section.")

    workbook_config = dict(model_params["workbook"])
    expected_sheets = [STOICHIOMETRIC_SHEET_NAME, COMPOSITION_SHEET_NAME, PARAMETER_SHEET_NAME]
    configured_sheets = list(workbook_config["sheets"])
    if configured_sheets != expected_sheets:
        raise ValueError("asm2d_tsn_simulation workbook sheets must match the required three-sheet contract.")

    dissolved_state_columns = list(workbook_config["dissolved_state_columns"])
    particulate_state_columns = list(workbook_config["particulate_state_columns"])
    state_columns = list(workbook_config["state_columns"])
    processes = list(workbook_config["processes"])
    parameter_rows = list(workbook_config["parameters"])
    state_units = dict(workbook_config["state_units"])

    _validate_unique_names(dissolved_state_columns, "dissolved_state_columns")
    _validate_unique_names(particulate_state_columns, "particulate_state_columns")
    _validate_unique_names(state_columns, "state_columns")

    if state_columns != dissolved_state_columns + particulate_state_columns:
        raise ValueError("asm2d_tsn_simulation state_columns must concatenate dissolved and particulate state columns.")

    missing_state_units = [state_name for state_name in state_columns if state_name not in state_units]
    if missing_state_units:
        missing_display = ", ".join(missing_state_units)
        raise KeyError(f"asm2d_tsn_simulation missing state units for: {missing_display}")

    if len(processes) != len(STOICHIOMETRIC_COEFFICIENTS):
        raise ValueError("asm2d_tsn_simulation workbook process count does not match the stoichiometric matrix definition.")

    process_indices = [int(process["index"]) for process in processes]
    if process_indices != list(range(1, len(processes) + 1)):
        raise ValueError("asm2d_tsn_simulation workbook processes must be sequentially indexed from 1.")

    parameter_names = [str(parameter_row["excel_name"]) for parameter_row in parameter_rows]
    _validate_unique_names(parameter_names, "parameter excel_name")

    for parameter_row in parameter_rows:
        for required_key in ("category", "symbol", "excel_name", "description", "value", "unit"):
            if required_key not in parameter_row:
                raise KeyError(f"asm2d_tsn_simulation parameter row missing '{required_key}'.")
        float(parameter_row["value"])

    return workbook_config


def _validate_solver_config(model_params: Mapping[str, Any]) -> dict[str, Any]:
    if "solver" not in model_params:
        raise KeyError("asm2d_tsn_simulation must define a solver section.")

    solver = dict(model_params["solver"])
    required_keys = (
        "lower_bound",
        "upper_bound",
        "initial_guess_floor",
        "warm_start_previous_weight",
        "warm_start_influent_weight",
        "initial_s_a_fraction",
        "initial_s_f_fraction",
        "initial_heterotroph_to_xs_ratio",
        "initial_pao_to_pp_ratio",
        "initial_aob_to_nh4_ratio",
        "initial_nob_to_no2_ratio",
        "multistart_s_a_fraction",
        "multistart_s_f_fraction",
        "multistart_heterotroph_to_xs_ratio",
        "multistart_pao_to_pp_ratio",
        "multistart_aob_to_nh4_ratio",
        "multistart_nob_to_no2_ratio",
        "dynamic_relaxation_days",
        "dynamic_absolute_tolerance",
        "dynamic_relative_tolerance",
        "dynamic_max_step",
        "residual_tolerance",
        "variable_tolerance",
        "gradient_tolerance",
        "acceptance_residual_max",
        "max_nfev",
    )

    for key in required_keys:
        if key not in solver:
            raise KeyError(f"asm2d_tsn_simulation solver missing '{key}'.")
        float(solver[key])

    lower_bound = float(solver["lower_bound"])
    upper_bound = float(solver["upper_bound"])
    initial_guess_floor = float(solver["initial_guess_floor"])
    if lower_bound < 0.0:
        raise ValueError("asm2d_tsn_simulation solver lower_bound must be non-negative.")
    if upper_bound <= lower_bound:
        raise ValueError("asm2d_tsn_simulation solver upper_bound must exceed lower_bound.")
    if not (lower_bound <= initial_guess_floor <= upper_bound):
        raise ValueError(
            "asm2d_tsn_simulation solver initial_guess_floor must lie between lower_bound and upper_bound."
        )

    previous_weight = float(solver["warm_start_previous_weight"])
    influent_weight = float(solver["warm_start_influent_weight"])
    if previous_weight < 0.0 or influent_weight < 0.0:
        raise ValueError("asm2d_tsn_simulation warm-start weights must be non-negative.")
    if previous_weight + influent_weight <= 0.0:
        raise ValueError("asm2d_tsn_simulation warm-start weights must sum to a positive value.")

    return solver


def _validate_runtime_structure(
    model_params: Mapping[str, Any],
    *,
    measured_output_columns: list[str] | None,
) -> dict[str, Any]:
    workbook_config = _validate_workbook_config(model_params)
    solver = _validate_solver_config(model_params)
    state_columns = list(workbook_config["state_columns"])
    if measured_output_columns is None:
        raise ValueError(
            "asm2d_tsn_simulation requires measured_output_columns derived from workbook composition_matrix."
        )
    measured_output_columns = [str(name) for name in measured_output_columns]
    process_names = [str(process["name"]) for process in workbook_config["processes"]]
    process_types = list(model_params["process_types"])
    operational_columns = list(model_params["operational_columns"])
    influent_state_ranges = dict(model_params["influent_state_ranges"])
    operational_ranges = dict(model_params["operational_ranges"])

    _validate_unique_names(measured_output_columns, "measured_output_columns")
    _validate_unique_names(process_names, "process names")
    _validate_unique_names(operational_columns, "operational_columns")

    if len(process_types) != len(process_names):
        raise ValueError("asm2d_tsn_simulation process_types must align with the configured process list.")

    missing_state_ranges = [state_name for state_name in state_columns if state_name not in influent_state_ranges]
    if missing_state_ranges:
        missing_display = ", ".join(missing_state_ranges)
        raise KeyError(f"asm2d_tsn_simulation missing influent_state_ranges for: {missing_display}")

    missing_operational_ranges = [name for name in operational_columns if name not in operational_ranges]
    if missing_operational_ranges:
        missing_display = ", ".join(missing_operational_ranges)
        raise KeyError(f"asm2d_tsn_simulation missing operational_ranges for: {missing_display}")

    return {
        "workbook_config": workbook_config,
        "solver": solver,
        "state_columns": state_columns,
        "measured_output_columns": measured_output_columns,
        "process_names": process_names,
        "process_types": process_types,
        "operational_columns": operational_columns,
        "influent_state_ranges": influent_state_ranges,
        "operational_ranges": operational_ranges,
    }


def _build_parameter_value_map(parameter_rows: list[Mapping[str, Any]]) -> dict[str, float]:
    return {str(parameter_row["excel_name"]): float(parameter_row["value"]) for parameter_row in parameter_rows}


def _evaluate_numeric_expression(expression: str | float | int, parameter_values: Mapping[str, float]) -> float:
    if isinstance(expression, (int, float)):
        return float(expression)
    formatted_expression = str(expression).format_map(parameter_values)
    return float(eval(formatted_expression, {"__builtins__": {}}, {}))


def _build_state_index(state_columns: list[str]) -> dict[str, int]:
    return {name: position for position, name in enumerate(state_columns)}


def _normalize_excel_reference(sheet_name: str, coordinate: str) -> str:
    return f"{str(sheet_name).strip().lower()}!{str(coordinate).replace('$', '').upper()}"


def _build_workbook_numeric_lookup(workbook) -> dict[str, float]:
    numeric_lookup: dict[str, float] = {}
    for worksheet in workbook.worksheets:
        for row in worksheet.iter_rows(
            min_row=1,
            max_row=worksheet.max_row,
            min_col=1,
            max_col=worksheet.max_column,
        ):
            for cell in row:
                cell_value = cell.value
                if isinstance(cell_value, bool):
                    continue
                if isinstance(cell_value, (int, float)):
                    numeric_lookup[_normalize_excel_reference(worksheet.title, cell.coordinate)] = float(cell_value)
    return numeric_lookup


def _evaluate_workbook_formula(
    formula: str,
    numeric_lookup: Mapping[str, float],
    *,
    current_sheet_name: str,
) -> float:
    expression = str(formula).strip()
    if expression.startswith("="):
        expression = expression[1:]

    def _replace_reference(match: re.Match[str]) -> str:
        sheet_name = match.group("sheet_quoted") or match.group("sheet_unquoted") or str(current_sheet_name)
        coordinate = f"{match.group('column').upper()}{match.group('row')}"
        lookup_key = _normalize_excel_reference(sheet_name, coordinate)
        if lookup_key not in numeric_lookup:
            raise KeyError(
                "Workbook formula references a non-numeric or missing cell: "
                f"{sheet_name}!{coordinate}."
            )
        return str(float(numeric_lookup[lookup_key]))

    resolved_expression = _EXCEL_REFERENCE_PATTERN.sub(_replace_reference, expression).replace("^", "**")
    return float(eval(resolved_expression, {"__builtins__": {}}, {}))


def _coerce_workbook_composition_value(
    cell_value: Any,
    numeric_lookup: Mapping[str, float],
    *,
    current_sheet_name: str,
) -> float:
    if cell_value is None:
        return 0.0
    if isinstance(cell_value, bool):
        return float(cell_value)
    if isinstance(cell_value, (int, float)):
        return float(cell_value)

    text_value = str(cell_value).strip()
    if not text_value:
        return 0.0
    if text_value.startswith("="):
        return _evaluate_workbook_formula(
            text_value,
            numeric_lookup,
            current_sheet_name=current_sheet_name,
        )

    try:
        return float(text_value)
    except ValueError as error:
        raise ValueError(f"Workbook composition_matrix contains a non-numeric value: {text_value!r}") from error


def _read_composition_matrix_from_workbook(
    workbook_path: str | Path,
    *,
    expected_state_columns: list[str],
) -> dict[str, Any]:
    workbook = load_workbook(filename=Path(workbook_path), data_only=False)
    try:
        if COMPOSITION_SHEET_NAME not in workbook.sheetnames:
            raise KeyError(f"Workbook is missing required sheet '{COMPOSITION_SHEET_NAME}'.")
        if PARAMETER_SHEET_NAME not in workbook.sheetnames:
            raise KeyError(f"Workbook is missing required sheet '{PARAMETER_SHEET_NAME}'.")

        worksheet = workbook[COMPOSITION_SHEET_NAME]
        header_values = [str(cell.value).strip() if cell.value is not None else "" for cell in worksheet[1]]
        if "state_variable" not in header_values:
            raise KeyError("Workbook composition_matrix must include a 'state_variable' header column.")

        state_column_number = header_values.index("state_variable") + 1
        reserved_headers = {"state_group", "state_variable", "unit"}
        composite_header_pairs = [
            (column_number, header_name)
            for column_number, header_name in enumerate(header_values, start=1)
            if header_name and header_name not in reserved_headers
        ]
        if not composite_header_pairs:
            raise ValueError("Workbook composition_matrix must define at least one composite output column.")

        measured_output_columns = [header_name for _, header_name in composite_header_pairs]
        _validate_unique_names(measured_output_columns, "composition_matrix output columns")

        numeric_lookup = _build_workbook_numeric_lookup(workbook)
        state_columns: list[str] = []
        coefficients_by_state: list[list[float]] = []
        for row_number in range(2, worksheet.max_row + 1):
            raw_state_name = worksheet.cell(row=row_number, column=state_column_number).value
            if raw_state_name is None:
                continue
            state_name = str(raw_state_name).strip()
            if not state_name:
                continue

            state_columns.append(state_name)
            row_coefficients: list[float] = []
            for column_number, _ in composite_header_pairs:
                raw_value = worksheet.cell(row=row_number, column=column_number).value
                row_coefficients.append(
                    _coerce_workbook_composition_value(
                        raw_value,
                        numeric_lookup,
                        current_sheet_name=worksheet.title,
                    )
                )
            coefficients_by_state.append(row_coefficients)

        if not state_columns:
            raise ValueError("Workbook composition_matrix must define at least one state_variable row.")
        _validate_unique_names(state_columns, "composition_matrix state_variable")

        if state_columns != expected_state_columns:
            raise ValueError(
                "Workbook composition_matrix state_variable rows must match configured workbook state_columns exactly and in order."
            )

        composition_matrix = np.asarray(coefficients_by_state, dtype=float).T
        expected_shape = (len(measured_output_columns), len(state_columns))
        if composition_matrix.shape != expected_shape:
            raise ValueError(
                "Workbook composition_matrix shape must match measured_output_columns x state_columns."
            )

        return {
            "state_columns": state_columns,
            "measured_output_columns": measured_output_columns,
            "composition_matrix": composition_matrix,
        }
    finally:
        workbook.close()


def _build_workbook_fingerprint(workbook_path: Path) -> dict[str, Any]:
    resolved_path = Path(workbook_path).resolve()
    stat_info = resolved_path.stat()
    return {
        "workbook_path": resolved_path.as_posix(),
        "workbook_sha256": compute_file_sha256(resolved_path),
        "workbook_mtime_ns": int(stat_info.st_mtime_ns),
        "workbook_size_bytes": int(stat_info.st_size),
    }


def _validate_cached_composition_payload(cache_payload: Mapping[str, Any]) -> None:
    for required_key in ("state_columns", "measured_output_columns", "composition_matrix"):
        if required_key not in cache_payload:
            raise KeyError(f"Cached composition payload is missing required key '{required_key}'.")


def load_asm2d_tsn_workbook_composition(
    *,
    repo_root: str | Path | None = None,
    workbook_path: str | Path | None = None,
    model_params: Mapping[str, Any] | None = None,
    paths_config: Mapping[str, Any] | None = None,
    use_cache: bool = True,
) -> dict[str, Any]:
    params = dict(model_params) if model_params is not None else load_asm2d_tsn_simulation_params(repo_root)
    workbook_config = _validate_workbook_config(params)
    expected_state_columns = list(workbook_config["state_columns"])
    resolved_workbook_path = (
        Path(workbook_path)
        if workbook_path is not None
        else resolve_asm2d_tsn_workbook_path(repo_root, paths_config=paths_config)
    )
    if not resolved_workbook_path.exists():
        raise FileNotFoundError(f"ASM2D-TSN workbook not found at {resolved_workbook_path}.")

    fingerprint = _build_workbook_fingerprint(resolved_workbook_path)
    cache_matrix_path, cache_metadata_path = resolve_asm2d_tsn_composition_cache_paths(
        fingerprint["workbook_sha256"],
        repo_root=repo_root,
        paths_config=paths_config,
    )

    if use_cache and cache_matrix_path.exists() and cache_metadata_path.exists():
        cache_metadata = load_json_file(cache_metadata_path)
        if (
            str(cache_metadata.get("workbook_sha256")) == str(fingerprint["workbook_sha256"])
            and int(cache_metadata.get("workbook_mtime_ns", -1)) == int(fingerprint["workbook_mtime_ns"])
            and int(cache_metadata.get("workbook_size_bytes", -1)) == int(fingerprint["workbook_size_bytes"])
        ):
            cache_payload = load_pickle_file(cache_matrix_path)
            _validate_cached_composition_payload(cache_payload)
            state_columns = [str(name) for name in cache_payload["state_columns"]]
            measured_output_columns = [str(name) for name in cache_payload["measured_output_columns"]]
            if state_columns != expected_state_columns:
                raise ValueError(
                    "Cached composition state_columns no longer match configured workbook state_columns."
                )
            composition_matrix = np.asarray(cache_payload["composition_matrix"], dtype=float)
            if composition_matrix.shape != (len(measured_output_columns), len(state_columns)):
                raise ValueError("Cached composition_matrix shape is invalid for cached schema.")
            return {
                "state_columns": state_columns,
                "measured_output_columns": measured_output_columns,
                "composition_matrix": composition_matrix,
                **fingerprint,
                "cache_source": "cache",
                "cache_paths": {
                    "composition_matrix": cache_matrix_path,
                    "composition_metadata": cache_metadata_path,
                },
            }

    parsed_composition = _read_composition_matrix_from_workbook(
        resolved_workbook_path,
        expected_state_columns=expected_state_columns,
    )
    composition_payload = {
        "state_columns": list(parsed_composition["state_columns"]),
        "measured_output_columns": list(parsed_composition["measured_output_columns"]),
        "composition_matrix": np.asarray(parsed_composition["composition_matrix"], dtype=float),
    }

    if use_cache:
        save_pickle_file(cache_matrix_path, composition_payload)
        save_json_file(
            cache_metadata_path,
            {
                "cache_schema_version": 1,
                "state_columns": composition_payload["state_columns"],
                "measured_output_columns": composition_payload["measured_output_columns"],
                **fingerprint,
            },
        )

    return {
        **composition_payload,
        **fingerprint,
        "cache_source": "workbook",
        "cache_paths": {
            "composition_matrix": cache_matrix_path,
            "composition_metadata": cache_metadata_path,
        },
    }


def get_asm2d_tsn_matrices(
    model_params: Mapping[str, Any] | None = None,
    *,
    repo_root: str | Path | None = None,
    paths_config: Mapping[str, Any] | None = None,
    use_composition_cache: bool = True,
) -> dict[str, Any]:
    params = dict(model_params) if model_params is not None else load_asm2d_tsn_simulation_params(repo_root)
    composition_bundle = load_asm2d_tsn_workbook_composition(
        repo_root=repo_root,
        model_params=params,
        paths_config=paths_config,
        use_cache=use_composition_cache,
    )
    measured_output_columns = list(composition_bundle["measured_output_columns"])
    runtime = _validate_runtime_structure(params, measured_output_columns=measured_output_columns)
    workbook_config = runtime["workbook_config"]
    parameter_values = _build_parameter_value_map(workbook_config["parameters"])
    state_columns = list(runtime["state_columns"])
    process_names = list(runtime["process_names"])
    process_types = list(runtime["process_types"])
    state_index = _build_state_index(state_columns)

    composition_state_columns = list(composition_bundle["state_columns"])
    if composition_state_columns != state_columns:
        raise ValueError(
            "Workbook composition_matrix state columns do not match the configured ASM2D-TSN state columns."
        )

    petersen_matrix = np.zeros((len(process_names), len(state_columns)), dtype=float)
    composition_matrix = np.asarray(composition_bundle["composition_matrix"], dtype=float)
    if composition_matrix.shape != (len(measured_output_columns), len(state_columns)):
        raise ValueError(
            "Workbook composition_matrix shape must match measured_output_columns x state_columns."
        )

    for row_index, process_definition in enumerate(STOICHIOMETRIC_COEFFICIENTS):
        row_values = petersen_matrix[row_index]
        direct_coefficients = process_definition["coefficients"]

        for state_name, expression in direct_coefficients.items():
            row_values[state_index[state_name]] = _evaluate_numeric_expression(expression, parameter_values)

        row_values[state_index["S_NH4"]] = -sum(
            row_values[state_index[state_name]] * _evaluate_numeric_expression(factor_expression, parameter_values)
            for state_name, factor_expression in NITROGEN_CONTINUITY_TERMS.items()
        )

        if "S_PO4" not in direct_coefficients:
            row_values[state_index["S_PO4"]] = -sum(
                row_values[state_index[state_name]] * _evaluate_numeric_expression(factor_expression, parameter_values)
                for state_name, factor_expression in PHOSPHORUS_CONTINUITY_TERMS.items()
            )

        row_values[state_index["S_ALK"]] = (
            row_values[state_index["S_NH4"]] / 14.0
            - row_values[state_index["S_NO2"]] / 14.0
            - row_values[state_index["S_NO3"]] / 14.0
            + row_values[state_index["S_PO4"]] / 31.0
        )

    return {
        "petersen_matrix": petersen_matrix,
        "composition_matrix": composition_matrix,
        "process_names": process_names,
        "process_types": process_types,
        "state_index": state_index,
        "state_columns": state_columns,
        "measured_output_columns": measured_output_columns,
        "composition_workbook_path": str(composition_bundle["workbook_path"]),
        "composition_workbook_sha256": str(composition_bundle["workbook_sha256"]),
        "composition_workbook_mtime_ns": int(composition_bundle["workbook_mtime_ns"]),
        "composition_workbook_size_bytes": int(composition_bundle["workbook_size_bytes"]),
        "composition_cache_source": str(composition_bundle["cache_source"]),
        "composition_cache_paths": dict(composition_bundle["cache_paths"]),
    }



@dataclass(frozen=True)
class SupervisedDatasetFrames:
    features: pd.DataFrame
    targets: pd.DataFrame
    constraint_reference: pd.DataFrame


@dataclass(frozen=True)
class DatasetSplit:
    features: pd.DataFrame
    targets: pd.DataFrame
    constraint_reference: pd.DataFrame


@dataclass(frozen=True)
class TrainTestDatasetSplits:
    train: DatasetSplit
    test: DatasetSplit


@dataclass(frozen=True)
class TrainTestSplitIndices:
    train: pd.Index
    test: pd.Index


def _ensure_columns_exist(frame: pd.DataFrame, required_columns: list[str]) -> None:
    missing_columns = [column_name for column_name in required_columns if column_name not in frame.columns]
    if missing_columns:
        missing_display = ", ".join(missing_columns)
        raise KeyError(f"Dataset is missing required columns: {missing_display}")


def _select_named_columns(frame: pd.DataFrame, required_columns: list[str]) -> pd.DataFrame:
    _ensure_columns_exist(frame, required_columns)
    first_positions: dict[str, int] = {}
    for position, column_name in enumerate(frame.columns):
        resolved_name = str(column_name)
        if resolved_name not in first_positions:
            first_positions[resolved_name] = int(position)

    selected_positions = [first_positions[str(column_name)] for column_name in required_columns]
    selected_frame = frame.iloc[:, selected_positions].copy()
    selected_frame.columns = required_columns
    return selected_frame


def build_fractional_input_fractional_output_dataset(
    dataset: pd.DataFrame,
    metadata: dict[str, Any],
    composition_matrix: np.ndarray,
) -> SupervisedDatasetFrames:
    state_columns = list(metadata["state_columns"])
    measured_output_columns = list(metadata["measured_output_columns"])
    operational_columns = list(metadata["operational_columns"])
    influent_state_columns = [f"In_{column_name}" for column_name in state_columns]
    target_columns = [f"Out_{column_name}" for column_name in state_columns]

    _ensure_columns_exist(dataset, operational_columns)
    _ensure_columns_exist(dataset, influent_state_columns)
    _ensure_columns_exist(dataset, target_columns)

    features = pd.concat(
        [
            _select_named_columns(dataset, operational_columns),
            _select_named_columns(dataset, influent_state_columns),
        ],
        axis=1,
    )
    targets = _select_named_columns(dataset, target_columns)
    constraint_reference = _select_named_columns(dataset, influent_state_columns).rename(
        columns=lambda column_name: column_name.replace("In_", "", 1)
    )

    if np.asarray(composition_matrix, dtype=float).shape != (len(measured_output_columns), len(state_columns)):
        raise ValueError(
            "composition_matrix shape must match measured_output_columns x state_columns for the fractional benchmark contract."
        )

    return SupervisedDatasetFrames(
        features=features,
        targets=targets,
        constraint_reference=constraint_reference,
    )


def build_icsor_supervised_dataset(
    dataset: pd.DataFrame,
    metadata: dict[str, Any],
    composition_matrix: np.ndarray,
) -> SupervisedDatasetFrames:
    return build_fractional_input_fractional_output_dataset(
        dataset,
        metadata,
        composition_matrix,
    )


def _split_frame(frame: pd.DataFrame, indices: pd.Index) -> pd.DataFrame:
    return frame.loc[indices].copy()


def _select_dataset_split(
    dataset: SupervisedDatasetFrames | DatasetSplit,
    indices: pd.Index,
) -> DatasetSplit:
    return DatasetSplit(
        features=_split_frame(dataset.features, indices),
        targets=_split_frame(dataset.targets, indices),
        constraint_reference=_split_frame(dataset.constraint_reference, indices),
    )


def make_train_test_split_indices(
    indices: pd.Index | np.ndarray | list[Any],
    *,
    test_fraction: float,
    random_seed: int,
) -> TrainTestSplitIndices:
    if not 0.0 < test_fraction < 1.0:
        raise ValueError("test_fraction must be between 0 and 1.")

    all_indices = pd.Index(indices)
    train_indices, test_indices = train_test_split(
        all_indices.to_numpy(),
        test_size=test_fraction,
        random_state=random_seed,
        shuffle=True,
    )

    return TrainTestSplitIndices(
        train=pd.Index(train_indices),
        test=pd.Index(test_indices),
    )


def apply_train_test_split_indices(
    supervised_dataset: SupervisedDatasetFrames | DatasetSplit,
    split_indices: TrainTestSplitIndices,
) -> TrainTestDatasetSplits:
    return TrainTestDatasetSplits(
        train=_select_dataset_split(supervised_dataset, split_indices.train),
        test=_select_dataset_split(supervised_dataset, split_indices.test),
    )


def sample_dataset_split_indices(
    indices: pd.Index | np.ndarray | list[Any],
    *,
    fraction: float,
    random_seed: int,
) -> pd.Index:
    if not 0.0 < fraction <= 1.0:
        raise ValueError("fraction must be between 0 and 1.")

    all_indices = pd.Index(indices)
    if fraction == 1.0:
        return all_indices.copy()

    sampled_indices, _ = train_test_split(
        all_indices.to_numpy(),
        train_size=fraction,
        random_state=random_seed,
        shuffle=True,
    )
    return pd.Index(sampled_indices)

def ensure_pip_available() -> None:
    pip_check_command = [sys.executable, "-m", "pip", "--version"]
    pip_check_result = subprocess.run(
        pip_check_command,
        capture_output=True,
        text=True,
    )
    if pip_check_result.returncode == 0:
        return

    print("pip is not available in this interpreter. Bootstrapping pip with ensurepip...")
    subprocess.run([sys.executable, "-m", "ensurepip", "--upgrade"], check=True)

    pip_check_result = subprocess.run(
        pip_check_command,
        capture_output=True,
        text=True,
    )
    if pip_check_result.returncode != 0:
        details = pip_check_result.stderr.strip() or pip_check_result.stdout.strip()
        raise RuntimeError(
            "pip is still unavailable after ensurepip completed. "
            f"Interpreter: {sys.executable}. Details: {details}"
        )


repo_root = get_repo_root()
pyproject_path = repo_root / "pyproject.toml"
pyproject = tomllib.loads(pyproject_path.read_text(encoding="utf-8"))

project_config = dict(pyproject.get("project", {}))
dependencies = list(project_config.get("dependencies", []))
if not dependencies:
    raise ValueError(f"No project.dependencies entries were found in {pyproject_path}")

uv_config = dict(pyproject.get("tool", {}).get("uv", {}))
extra_index_urls = list(
    dict.fromkeys(
        entry["url"]
        for entry in uv_config.get("index", [])
        if isinstance(entry, dict) and entry.get("url")
    )
)

ensure_pip_available()

install_command = [sys.executable, "-m", "pip", "install", "--upgrade"]
for index_url in extra_index_urls:
    install_command.extend(["--extra-index-url", index_url])
install_command.extend(dependencies)

print(f"Installing {len(dependencies)} dependencies from {pyproject_path.name}.")
print(f"Python executable: {sys.executable}")
if project_config.get("requires-python"):
    print(f"Project requires Python {project_config['requires-python']}")
if extra_index_urls:
    print("Using additional package indexes:")
    for index_url in extra_index_urls:
        print(f"- {index_url}")

subprocess.run(install_command, check=True, cwd=repo_root)

print("Dependency installation complete.")
print("Restart the kernel if any newly installed packages are not immediately available.")

In [ ]:
import json

import numpy as np
import pandas as pd
from IPython.display import display as ipython_display

repo_root = get_repo_root()
simulation_dir = repo_root / "data" / "asm2d-tsn" / "simulation"

dataset_candidates = {
    path.stem.removeprefix("data_"): path
    for path in simulation_dir.glob("data_*.csv")
}
metadata_candidates = {
    path.stem.removeprefix("metadata_"): path
    for path in simulation_dir.glob("metadata_*.json")
}

if not dataset_candidates:
    raise FileNotFoundError(f"No ASM2D-TSN datasets found in {simulation_dir}")
if not metadata_candidates:
    raise FileNotFoundError(f"No ASM2D-TSN metadata files found in {simulation_dir}")

shared_timestamps = sorted(set(dataset_candidates) & set(metadata_candidates))
if shared_timestamps:
    latest_timestamp = shared_timestamps[-1]
    latest_dataset_path = dataset_candidates[latest_timestamp]
    latest_metadata_path = metadata_candidates[latest_timestamp]
else:
    latest_dataset_path = max(dataset_candidates.values(), key=lambda path: path.stat().st_mtime)
    latest_metadata_path = max(metadata_candidates.values(), key=lambda path: path.stat().st_mtime)

dataset = pd.read_csv(latest_dataset_path)
metadata = json.loads(latest_metadata_path.read_text(encoding="utf-8-sig"))
artifact_paths = {
    "dataset_csv": latest_dataset_path,
    "metadata_json": latest_metadata_path,
}

model_params = load_asm2d_tsn_simulation_params(repo_root)
matrix_bundle = get_asm2d_tsn_matrices(model_params, repo_root=repo_root)
state_columns = list(matrix_bundle["state_columns"])
measured_output_columns = list(matrix_bundle["measured_output_columns"])
petersen_matrix = np.asarray(matrix_bundle["petersen_matrix"], dtype=float)
composition_matrix = np.asarray(matrix_bundle["composition_matrix"], dtype=float)

metadata_state_columns = list(metadata.get("state_columns", []))
metadata_measured_output_columns = list(metadata.get("measured_output_columns", []))
if metadata_state_columns and metadata_state_columns != state_columns:
    raise ValueError(
        "Metadata state_columns do not match workbook-derived state columns. Regenerate simulation artifacts."
    )
if metadata_measured_output_columns and metadata_measured_output_columns != measured_output_columns:
    raise ValueError(
        "Metadata measured_output_columns do not match workbook composition_matrix columns. Regenerate simulation artifacts."
    )

metadata_table = pd.DataFrame(
    {
        "field": list(metadata.keys()),
        "value": [
            json.dumps(value)
            if isinstance(value, (dict, list))
            else None
            if value is None
            else str(value)
            for value in metadata.values()
        ],
    }
)

matrix_source = matrix_bundle["composition_workbook_path"]
print(f"Loaded {len(dataset)} rows for {metadata['simulation_name']}.")
print(f"Dataset loaded from: {artifact_paths['dataset_csv']}")
print(f"Metadata loaded from: {artifact_paths['metadata_json']}")
print(f"Matrix source: {matrix_source}")
print(f"Composition cache source: {matrix_bundle['composition_cache_source']}")
print(f"Petersen matrix shape: {petersen_matrix.shape}")
print(f"Composition matrix shape: {composition_matrix.shape}")

ipython_display(dataset.head())
ipython_display(metadata_table)
ipython_display(pd.DataFrame(petersen_matrix, index=metadata["processes"], columns=state_columns))
ipython_display(pd.DataFrame(composition_matrix, index=measured_output_columns, columns=state_columns))

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display as ipython_display
from scipy.linalg import null_space

from src.models.ml.adaboost_regressor import (
    build_adaboost_regressor_model,
    load_adaboost_regressor_params,
    run_adaboost_regressor_pipeline,
)
from src.models.ml.ann_deep_regressor import (
    build_ann_deep_regressor_model,
    load_ann_deep_regressor_params,
    run_ann_deep_regressor_pipeline,
)
from src.models.ml.catboost_regressor import (
    build_catboost_regressor_model,
    load_catboost_regressor_params,
    run_catboost_regressor_pipeline,
)
from src.models.ml.icsor_coupled_qp import (
    load_icsor_coupled_qp_params,
    run_icsor_coupled_qp_pipeline,
)
from src.models.ml.knn_regressor import (
    build_knn_regressor_model,
    load_knn_regressor_params,
    run_knn_regressor_pipeline,
)
from src.models.ml.lightgbm_regressor import (
    build_lightgbm_regressor_model,
    load_lightgbm_regressor_params,
    run_lightgbm_regressor_pipeline,
)
from src.models.ml.pls_regressor import (
    build_pls_regressor_model,
    load_pls_regressor_params,
    run_pls_regressor_pipeline,
)
from src.models.ml.random_forest_regressor import (
    build_random_forest_regressor_model,
    load_random_forest_regressor_params,
    run_random_forest_regressor_pipeline,
)
from src.models.ml.svr_regressor import (
    build_svr_regressor_model,
    load_svr_regressor_params,
    run_svr_regressor_pipeline,
)
from src.models.ml.xgboost_regressor import (
    build_xgboost_regressor_model,
    load_xgboost_regressor_params,
    run_xgboost_regressor_pipeline,
)
from src.utils.analysis import (
    add_effective_metric_columns,
    collate_model_analysis_results,
    rank_metric_summary,
    run_model_dataset_size_analysis,
    summarize_metric_distribution,
)
from src.utils.plot import (
    PIBRE_THEME_TOKENS,
    plot_coefficient_heatmap,
    plot_icsor_target_atlas,
    plot_metric_summary_lines,
    save_figure_pdf,
)
from src.utils.train import tune_icsor_coupled_qp_hyperparameters, tune_tabular_regressor_hyperparameters

repo_root = get_repo_root()
paths_config = load_paths_config(repo_root)
submission_asset_dir = repo_root / paths_config["submission_asset_dir"]
submission_asset_dir.mkdir(parents=True, exist_ok=True)

comparison_table_pattern = paths_config["comparison_notebook_table_pattern"]
comparison_figure_pattern = paths_config["comparison_notebook_figure_pattern"]

required_assets = [
    "table_results_benchmark_sample.csv",
    "table_results_target_sample.csv",
    "figure2_rmse_learning_curve.pdf",
    "table_results_scaling_sample.csv",
    "figure3_runtime_learning_curve.pdf",
    "table_results_physical_sample.csv",
    "figure4_icsor_structure.pdf",
    "table_results_icsor_regularization_sample.csv",
    "figureS1_cod_icsor_structure.pdf",
    "figureS2_tn_icsor_structure.pdf",
    "figureS3_tp_icsor_structure.pdf",
    "figureS4_tss_icsor_structure.pdf",
]

print(f"Submission asset directory: {submission_asset_dir}")
print(f"Comparison table pattern: {comparison_table_pattern}")
print(f"Comparison figure pattern: {comparison_figure_pattern}")
print(f"Expected asset count: {len(required_assets)}")


In [ ]:
analysis_metric = "projected_MSE"

ml_orchestration_params = load_ml_orchestration_params()
ml_orchestration = dict(ml_orchestration_params["hyperparameters"])
analysis_settings = dict(ml_orchestration_params.get("analysis", {}))
benchmark_settings = dict(ml_orchestration_params.get("benchmark", {}))
USE_OPTUNA = bool(benchmark_settings.get("use_optuna", False))

configured_analysis_overrides = {
    "min_total_samples": int(analysis_settings.get("min_total_samples", 500)),
    "max_total_samples": int(analysis_settings.get("max_total_samples", 10000)),
    "total_sample_step": int(analysis_settings.get("total_sample_step", 950)),
    "n_repeats": int(analysis_settings.get("n_repeats", 100)),
    "test_fraction": float(analysis_settings.get("test_fraction", ml_orchestration.get("test_fraction", 0.2))),
    "random_seed": int(ml_orchestration.get("random_seed", 42)),
}

# Full implementation mode: always use repository-configured analysis scales.
FAST_E2E_TEST = False
analysis_overrides = configured_analysis_overrides

print(f"Classical benchmark Optuna enabled: {USE_OPTUNA}")
print(f"Analysis metric set to: {analysis_metric}")
print(f"Fast end-to-end test mode: {FAST_E2E_TEST}")
print("Repeated-size analysis overrides:")
for key, value in analysis_overrides.items():
    print(f"- {key}: {value}")

In [ ]:
repo_root = get_repo_root()
simulation_dir = repo_root / "data" / "asm2d-tsn" / "simulation"

dataset_candidates = {
    path.stem.removeprefix("data_"): path
    for path in simulation_dir.glob("data_*.csv")
}
metadata_candidates = {
    path.stem.removeprefix("metadata_"): path
    for path in simulation_dir.glob("metadata_*.json")
}

if not dataset_candidates:
    raise FileNotFoundError(f"No ASM2D-TSN datasets found in {simulation_dir}")
if not metadata_candidates:
    raise FileNotFoundError(f"No ASM2D-TSN metadata files found in {simulation_dir}")

shared_timestamps = sorted(set(dataset_candidates) & set(metadata_candidates))
if shared_timestamps:
    latest_timestamp = shared_timestamps[-1]
    latest_dataset_path = dataset_candidates[latest_timestamp]
    latest_metadata_path = metadata_candidates[latest_timestamp]
else:
    latest_dataset_path = max(dataset_candidates.values(), key=lambda path: path.stat().st_mtime)
    latest_metadata_path = max(metadata_candidates.values(), key=lambda path: path.stat().st_mtime)

dataset = pd.read_csv(latest_dataset_path)
metadata = json.loads(latest_metadata_path.read_text(encoding="utf-8-sig"))
artifact_paths = {
    "dataset_csv": latest_dataset_path,
    "metadata_json": latest_metadata_path,
}

model_params = load_asm2d_tsn_simulation_params(repo_root)
matrix_bundle = get_asm2d_tsn_matrices(model_params, repo_root=repo_root)
state_columns = list(matrix_bundle["state_columns"])
measured_output_columns = list(matrix_bundle["measured_output_columns"])
petersen_matrix = np.asarray(matrix_bundle["petersen_matrix"], dtype=float)
composition_matrix = np.asarray(matrix_bundle["composition_matrix"], dtype=float)

metadata_state_columns = list(metadata.get("state_columns", []))
metadata_measured_output_columns = list(metadata.get("measured_output_columns", []))
if metadata_state_columns and metadata_state_columns != state_columns:
    raise ValueError(
        "Metadata state_columns do not match workbook-derived state columns. Regenerate simulation artifacts."
    )
if metadata_measured_output_columns and metadata_measured_output_columns != measured_output_columns:
    raise ValueError(
        "Metadata measured_output_columns do not match workbook composition_matrix columns. Regenerate simulation artifacts."
    )

print(f"Loaded {len(dataset)} rows for {metadata['simulation_name']}.")
print(f"Dataset loaded from: {artifact_paths['dataset_csv']}")
print(f"Metadata loaded from: {artifact_paths['metadata_json']}")
print(f"Matrix source: {matrix_bundle['composition_workbook_path']}")
print(f"Composition cache source: {matrix_bundle['composition_cache_source']}")
print(f"Petersen matrix shape: {petersen_matrix.shape}")
print(f"Composition matrix shape: {composition_matrix.shape}")

ipython_display(dataset.head())

In [ ]:
icsor_constraint_basis = null_space(petersen_matrix)
icsor_A_matrix = icsor_constraint_basis.T

icsor_A_matrix = np.round(icsor_A_matrix, 5)
icsor_A_matrix[np.abs(icsor_A_matrix) < 1e-10] = 0.0

for row_index in range(icsor_A_matrix.shape[0]):
    non_zero_entries = icsor_A_matrix[row_index, icsor_A_matrix[row_index, :] != 0]
    if len(non_zero_entries) > 0:
        icsor_A_matrix[row_index, :] = icsor_A_matrix[row_index, :] / non_zero_entries[0]

A_matrix = icsor_A_matrix.copy()

print(f"Fractional Petersen matrix shape: {petersen_matrix.shape}")
print(f"Invariant matrix shape: {A_matrix.shape}")

In [ ]:
classical_benchmark_dataset = build_fractional_input_fractional_output_dataset(
    dataset,
    metadata,
    composition_matrix,
)
icsor_dataset = build_icsor_supervised_dataset(
    dataset,
    metadata,
    composition_matrix,
)

shared_split_indices = make_train_test_split_indices(
    dataset.index,
    test_fraction=float(ml_orchestration["test_fraction"]),
    random_seed=int(ml_orchestration["random_seed"]),
)
main_dataset_splits = apply_train_test_split_indices(
    classical_benchmark_dataset,
    shared_split_indices,
)
icsor_dataset_splits = apply_train_test_split_indices(
    icsor_dataset,
    shared_split_indices,
)

optuna_indices = sample_dataset_split_indices(
    shared_split_indices.train,
    fraction=float(ml_orchestration["optuna_dataset_fraction"]),
    random_seed=int(ml_orchestration["random_seed"]),
)
tuning_split_indices = make_train_test_split_indices(
    optuna_indices,
    test_fraction=float(ml_orchestration["optuna_test_fraction"]),
    random_seed=int(ml_orchestration["random_seed"]),
)
tuning_dataset_splits = apply_train_test_split_indices(
    classical_benchmark_dataset,
    tuning_split_indices,
)
icsor_tuning_dataset_splits = apply_train_test_split_indices(
    icsor_dataset,
    tuning_split_indices,
)

print("Split contract ready.")
print(f"Main train size: {len(main_dataset_splits.train.features)}")
print(f"Main test size: {len(main_dataset_splits.test.features)}")
print(f"Optuna train size: {len(tuning_dataset_splits.train.features)}")
print(f"Optuna test size: {len(tuning_dataset_splits.test.features)}")

In [ ]:
def comparison_table_path(artifact_name: str) -> Path:
    return repo_root / comparison_table_pattern.format(artifact_name=artifact_name)


def comparison_figure_path(artifact_name: str) -> Path:
    return repo_root / comparison_figure_pattern.format(artifact_name=artifact_name)


def export_table_csv(artifact_name: str, frame: pd.DataFrame, *, index: bool = False) -> Path:
    export_path = comparison_table_path(artifact_name)
    export_path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(export_path, index=index)
    print(f"Saved table: {export_path}")
    return export_path


def export_figure_pdf(artifact_name: str, figure: plt.Figure) -> Path:
    export_path = comparison_figure_path(artifact_name)
    save_figure_pdf(figure, export_path)
    print(f"Saved figure: {export_path}")
    return export_path


retention_fraction = 1e-4

retained_model_specs = {
    "icsor": {
        "label": "ICSOR",
        "family": "Physics-informed",
        "dataset_key": "icsor",
        "load_params": load_icsor_coupled_qp_params,
        "runner": run_icsor_coupled_qp_pipeline,
        "tune": tune_icsor_coupled_qp_hyperparameters,
        "default_hyperparameter_overrides": {
            "training_method": "recursive_qp",
            "objective": "recursive_qp",
        },
        "extra_runner_kwargs": {
            "composition_matrix": composition_matrix,
            "measured_output_columns": list(metadata["measured_output_columns"]),
            "composition_source": metadata.get("composition_source"),
        },
    },
    "xgboost_regressor": {
        "label": "XGBoost",
        "family": "Classical",
        "dataset_key": "classical",
        "load_params": load_xgboost_regressor_params,
        "build_model": build_xgboost_regressor_model,
        "runner": run_xgboost_regressor_pipeline,
    },
    "lightgbm_regressor": {
        "label": "LightGBM",
        "family": "Classical",
        "dataset_key": "classical",
        "load_params": load_lightgbm_regressor_params,
        "build_model": build_lightgbm_regressor_model,
        "runner": run_lightgbm_regressor_pipeline,
    },
    "catboost_regressor": {
        "label": "CatBoost",
        "family": "Classical",
        "dataset_key": "classical",
        "load_params": load_catboost_regressor_params,
        "build_model": build_catboost_regressor_model,
        "runner": run_catboost_regressor_pipeline,
    },
    "adaboost_regressor": {
        "label": "AdaBoost",
        "family": "Classical",
        "dataset_key": "classical",
        "load_params": load_adaboost_regressor_params,
        "build_model": build_adaboost_regressor_model,
        "runner": run_adaboost_regressor_pipeline,
    },
    "random_forest_regressor": {
        "label": "Random Forest",
        "family": "Classical",
        "dataset_key": "classical",
        "load_params": load_random_forest_regressor_params,
        "build_model": build_random_forest_regressor_model,
        "runner": run_random_forest_regressor_pipeline,
    },
    "svr_regressor": {
        "label": "SVR",
        "family": "Classical",
        "dataset_key": "classical",
        "load_params": load_svr_regressor_params,
        "build_model": build_svr_regressor_model,
        "runner": run_svr_regressor_pipeline,
    },
    "knn_regressor": {
        "label": "k-NN",
        "family": "Classical",
        "dataset_key": "classical",
        "load_params": load_knn_regressor_params,
        "build_model": build_knn_regressor_model,
        "runner": run_knn_regressor_pipeline,
    },
    "pls_regressor": {
        "label": "PLS",
        "family": "Classical",
        "dataset_key": "classical",
        "load_params": load_pls_regressor_params,
        "build_model": build_pls_regressor_model,
        "runner": run_pls_regressor_pipeline,
    },
    "ann_deep_regressor": {
        "label": "MLP",
        "family": "Classical",
        "dataset_key": "classical",
        "load_params": load_ann_deep_regressor_params,
        "build_model": build_ann_deep_regressor_model,
        "runner": run_ann_deep_regressor_pipeline,
    },
}

icsor_spec = retained_model_specs["icsor"]
icsor_loader_name = icsor_spec["load_params"].__name__
icsor_runner_name = icsor_spec["runner"].__name__
icsor_tuner_name = icsor_spec["tune"].__name__
if (
    icsor_loader_name != "load_icsor_coupled_qp_params"
    or icsor_runner_name != "run_icsor_coupled_qp_pipeline"
    or icsor_tuner_name != "tune_icsor_coupled_qp_hyperparameters"
):
    raise RuntimeError(
        "ICSOR retained model must use coupled-QP entry points "
        "(load_icsor_coupled_qp_params, run_icsor_coupled_qp_pipeline, tune_icsor_coupled_qp_hyperparameters)."
    )

icsor_model_params = icsor_spec["load_params"]()
icsor_training_defaults = dict(icsor_model_params.get("training_defaults", {}))
icsor_default_training_method = str(icsor_training_defaults.get("training_method", "")).strip().lower()
if icsor_default_training_method != "recursive_qp":
    raise RuntimeError(
        "ICSOR coupled-QP defaults must set training_method='recursive_qp' in config/params.json."
    )

print(
    "ICSOR implementation locked to coupled-QP: "
    "loader=load_icsor_coupled_qp_params, "
    "runner=run_icsor_coupled_qp_pipeline, "
    "tuner=tune_icsor_coupled_qp_hyperparameters"
)
print(f"Retained model count: {len(retained_model_specs)}")
print("Retained model labels:")
for spec in retained_model_specs.values():
    print(f"- {spec['label']}")

In [ ]:
import time

selected_hyperparameters: dict[str, dict[str, object]] = {}
optuna_summaries: dict[str, dict[str, object] | None] = {}
fixed_split_results: dict[str, dict[str, object]] = {}
fixed_split_runtime_rows: list[dict[str, object]] = []

for model_name, spec in retained_model_specs.items():
    print(f"Running fixed-split benchmark for {spec['label']}...")
    params = spec["load_params"]()
    hyperparameters = dict(params["training_defaults"])
    hyperparameters.update(dict(spec.get("default_hyperparameter_overrides", {})))
    optuna_summary = None

    if USE_OPTUNA and bool(params.get("search_space")):
        if model_name == "icsor":
            hyperparameters, optuna_summary = spec["tune"](
                icsor_tuning_dataset_splits.train,
                icsor_tuning_dataset_splits.test,
                A_matrix=icsor_A_matrix,
                composition_matrix=composition_matrix,
                measured_output_columns=list(metadata["measured_output_columns"]),
                model_params=params,
                model_hyperparameters=hyperparameters,
                n_trials=int(ml_orchestration["n_trials"]),
                timeout=ml_orchestration.get("timeout_seconds"),
                show_progress_bar=True,
            )
        else:
            hyperparameters, optuna_summary = tune_tabular_regressor_hyperparameters(
                model_name,
                spec["build_model"],
                tuning_dataset_splits.train,
                tuning_dataset_splits.test,
                A_matrix=A_matrix,
                composition_matrix=composition_matrix,
                measured_output_columns=list(metadata["measured_output_columns"]),
                model_params=params,
                n_trials=int(ml_orchestration["n_trials"]),
                timeout=ml_orchestration.get("timeout_seconds"),
                show_progress_bar=True,
            )

    hyperparameters.update(dict(spec.get("default_hyperparameter_overrides", {})))
    selected_hyperparameters[model_name] = dict(hyperparameters)
    optuna_summaries[model_name] = optuna_summary

    dataset_splits = icsor_dataset_splits if spec["dataset_key"] == "icsor" else main_dataset_splits
    constraint_matrix = icsor_A_matrix if spec["dataset_key"] == "icsor" else A_matrix

    runner_kwargs = {
        "composition_matrix": composition_matrix,
        "measured_output_columns": list(metadata["measured_output_columns"]),
        "repo_root": repo_root,
        "model_params": params,
        "model_hyperparameters": hyperparameters,
        "optuna_summary": optuna_summary,
        "persist_artifacts": False,
        "show_progress": False,
    }
    runner_kwargs.update(dict(spec.get("extra_runner_kwargs", {})))

    run_started_at = time.perf_counter()
    result = spec["runner"](
        dataset_splits.train,
        dataset_splits.test,
        constraint_matrix,
        **runner_kwargs,
    )
    run_elapsed_seconds = float(time.perf_counter() - run_started_at)

    if model_name == "icsor":
        resolved_bundle_model_name = str(result.get("model_bundle", {}).get("model_name", ""))
        resolved_training_method = str(result.get("model_bundle", {}).get("training_method", "")).strip().lower()
        if resolved_bundle_model_name != "icsor_coupled_qp" or resolved_training_method != "recursive_qp":
            raise RuntimeError(
                "ICSOR run did not execute the coupled-QP recursive_qp path. "
                f"Observed model_name={resolved_bundle_model_name}, training_method={resolved_training_method}."
            )
        print(
            "ICSOR coupled-QP verification: "
            f"model_name={resolved_bundle_model_name}, training_method={resolved_training_method}"
        )

    fixed_split_results[model_name] = result
    fixed_split_runtime_rows.append(
        {
            "model_name": model_name,
            "model_label": spec["label"],
            "runtime_seconds": run_elapsed_seconds,
        }
    )

fixed_split_runtime_summary = pd.DataFrame(fixed_split_runtime_rows).sort_values("model_label").reset_index(drop=True)
print("Fixed-split benchmark complete for retained models.")
ipython_display(fixed_split_runtime_summary)

In [ ]:
analysis_results_by_model: dict[str, dict[str, object]] = {}

for model_name, spec in retained_model_specs.items():
    print(f"Running repeated-size analysis for {spec['label']}...")

    dataset = icsor_dataset if spec["dataset_key"] == "icsor" else classical_benchmark_dataset
    constraint_matrix = icsor_A_matrix if spec["dataset_key"] == "icsor" else A_matrix

    runner_kwargs = {
        "composition_matrix": composition_matrix,
        "measured_output_columns": list(metadata["measured_output_columns"]),
    }
    runner_kwargs.update(dict(spec.get("extra_runner_kwargs", {})))

    analysis_result = run_model_dataset_size_analysis(
        model_name,
        dataset,
        constraint_matrix,
        spec["runner"],
        model_params=spec["load_params"](),
        model_hyperparameters=selected_hyperparameters[model_name],
        repo_root=repo_root,
        show_progress=True,
        show_runner_progress=False,
        persist_artifacts=False,
        include_prediction_tables=False,
        extra_runner_kwargs=runner_kwargs,
        **analysis_overrides,
    )
    analysis_results_by_model[model_name] = analysis_result

print("Repeated-size analysis complete for retained models.")

In [ ]:
import importlib
import src.utils.analysis as analysis_module
importlib.reload(analysis_module)
from src.utils.analysis import collate_model_analysis_results

comparison_model_labels = {
    model_name: retained_model_specs[model_name]["label"]
    for model_name in retained_model_specs
}
comparison_model_families = {
    model_name: retained_model_specs[model_name]["family"]
    for model_name in retained_model_specs
}

comparison_bundle = collate_model_analysis_results(
    analysis_results_by_model,
    model_labels=comparison_model_labels,
    model_families=comparison_model_families,
)

comparison_metric_basenames = ["RMSE", "MAE", "R2", "MAPE", "MSE"]
comparison_primary_metric = "effective_RMSE"
comparison_primary_split = "test"
comparison_base_model_columns = ["model_name", "model_label", "model_family", "model_order"]
comparison_run_metadata = comparison_bundle["run_metadata"].copy()
comparison_effective_aggregate_metrics = comparison_bundle["effective_aggregate_metrics"].copy()
comparison_per_target_metrics = comparison_bundle["per_target_metrics"].copy()
comparison_prediction_diagnostics = comparison_bundle["prediction_diagnostics"].copy()
comparison_prediction_target_diagnostics = comparison_bundle["prediction_target_diagnostics"].copy()
comparison_smallest_train_size = int(comparison_run_metadata["train_size"].min())
comparison_largest_train_size = int(comparison_run_metadata["train_size"].max())
comparison_target_labels = [
    str(target_name).removeprefix("Out_")
    for target_name in metadata["measured_output_columns"]
]

print("Comparison bundles derived.")
print(f"Train-size range: {comparison_smallest_train_size} to {comparison_largest_train_size}")
print(f"Primary metric: {comparison_primary_metric}")

In [ ]:
# Ensure coefficient payload is available for both legacy ICSOR and coupled-QP bundles.
icsor_model_bundle = fixed_split_results["icsor"]["model_bundle"]

if "effective_coefficients" not in icsor_model_bundle:
    design_schema = dict(icsor_model_bundle["design_schema"])
    coefficient_matrix = np.asarray(icsor_model_bundle["B_matrix"], dtype=float).T

    def _block_slice(schema: dict[str, object], block_name: str) -> slice:
        block_range = dict(schema["block_ranges"][block_name])
        return slice(int(block_range["start"]), int(block_range["stop"]))

    operational_dim = int(design_schema["dimensions"]["operational"])
    influent_dim = int(design_schema["dimensions"]["influent"])
    target_dim = int(coefficient_matrix.shape[1])

    linear_operational = coefficient_matrix[_block_slice(design_schema, "linear_operational"), :].T
    linear_influent = coefficient_matrix[_block_slice(design_schema, "linear_influent"), :].T

    bias_slice = _block_slice(design_schema, "bias")
    if bias_slice.stop > bias_slice.start:
        bias_vector = coefficient_matrix[bias_slice, :].reshape(-1)
    else:
        bias_vector = np.zeros(target_dim, dtype=float)

    theta_uu = coefficient_matrix[_block_slice(design_schema, "quadratic_operational"), :].T.reshape(
        target_dim,
        operational_dim,
        operational_dim,
    )
    theta_cc = coefficient_matrix[_block_slice(design_schema, "quadratic_influent"), :].T.reshape(
        target_dim,
        influent_dim,
        influent_dim,
    )
    theta_uc = coefficient_matrix[_block_slice(design_schema, "interaction_operational_influent"), :].T.reshape(
        target_dim,
        operational_dim,
        influent_dim,
    )

    icsor_model_bundle["effective_coefficients"] = {
        "W_u": linear_operational,
        "W_in": linear_influent,
        "b": bias_vector,
        "Theta_uu": theta_uu,
        "Theta_cc": theta_cc,
        "Theta_uc": theta_uc,
    }

print("ICSOR coefficient payload ready for interpretability cell.")

In [ ]:
comparison_test_aggregate_metrics = comparison_effective_aggregate_metrics.loc[
    comparison_effective_aggregate_metrics["split_name"] == "test"
].copy()
comparison_test_per_target_metrics = comparison_per_target_metrics.loc[
    comparison_per_target_metrics["split_name"] == "test"
].copy()

benchmark_rows = []
for model_name, spec in retained_model_specs.items():
    report_metrics = add_effective_metric_columns(fixed_split_results[model_name]["test_report"]["aggregate_metrics"])
    preferred_type = "projected" if "projected" in set(report_metrics["prediction_type"]) else "raw"
    effective_row = report_metrics.loc[report_metrics["prediction_type"] == preferred_type].iloc[0]
    benchmark_rows.append(
        {
            "Model": spec["label"],
            "RMSE": float(effective_row["effective_RMSE"]),
            "MAE": float(effective_row["effective_MAE"]),
            "R2": float(effective_row["effective_R2"]),
            "MAPE": float(effective_row["effective_MAPE"]),
        }
    )

benchmark_table = pd.DataFrame(benchmark_rows).sort_values("RMSE").reset_index(drop=True)
model_display_order = benchmark_table["Model"].tolist()

rmse_target_summary = summarize_metric_distribution(
    comparison_test_per_target_metrics,
    metric_name="effective_RMSE",
    group_columns=[*comparison_base_model_columns, "train_size", "target"],
)
rmse_target_largest = rmse_target_summary.loc[
    rmse_target_summary["train_size"] == comparison_largest_train_size
].copy()
rmse_target_largest["target"] = rmse_target_largest["target"].astype(str).str.removeprefix("Out_")

target_table = (
    rmse_target_largest
    .pivot(index="model_label", columns="target", values="metric_mean")
    .reindex(index=model_display_order, columns=comparison_target_labels)
    .reset_index()
    .rename(columns={"model_label": "Model"})
)

def curve_nauc(summary_frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for model_label, group in summary_frame.groupby("model_label", dropna=False):
        group = group.sort_values("train_size")
        x_values = group["train_size"].to_numpy(dtype=float)
        y_values = group["metric_mean"].to_numpy(dtype=float)
        if len(group) == 1:
            auc_value = float(y_values[0])
        else:
            auc_value = float((np.trapezoid if hasattr(np, "trapezoid") else np.trapz)(y_values, x_values) / (x_values.max() - x_values.min()))
        rows.append({"Model": str(model_label), "RMSE nAUC": auc_value})
    return pd.DataFrame(rows)

rmse_learning_curve_summary = summarize_metric_distribution(
    comparison_test_aggregate_metrics,
    metric_name="effective_RMSE",
    group_columns=[*comparison_base_model_columns, "train_size"],
)
final_rmse_table = (
    rmse_learning_curve_summary.loc[rmse_learning_curve_summary["train_size"] == comparison_largest_train_size]
    .loc[:, ["model_label", "metric_mean"]]
    .rename(columns={"model_label": "Model", "metric_mean": "Final RMSE"})
)
rmse_nauc_table = curve_nauc(rmse_learning_curve_summary)

rank_frames = []
for metric_basename in comparison_metric_basenames:
    metric_name = f"effective_{metric_basename}"
    metric_target_summary = summarize_metric_distribution(
        comparison_test_per_target_metrics,
        metric_name=metric_name,
        group_columns=[*comparison_base_model_columns, "train_size", "target"],
    )
    ranked_frame = rank_metric_summary(
        metric_target_summary,
        group_columns=["train_size", "target"],
        metric_name=metric_name,
    )
    rank_frames.append(
        ranked_frame.groupby(["model_name", "model_label", "model_family", "model_order"], dropna=False)["metric_rank"]
        .mean()
        .reset_index(name=f"{metric_basename}_average_rank")
    )

sample_rank_table = rank_frames[0]
for frame in rank_frames[1:]:
    sample_rank_table = sample_rank_table.merge(
        frame,
        on=["model_name", "model_label", "model_family", "model_order"],
        how="outer",
    )
rank_columns = [f"{metric_basename}_average_rank" for metric_basename in comparison_metric_basenames]
sample_rank_table["Sample Avg. Rank"] = sample_rank_table[rank_columns].mean(axis=1)
sample_rank_table = sample_rank_table.loc[:, ["model_label", "Sample Avg. Rank"]].rename(columns={"model_label": "Model"})

rmse_train_test_summary = summarize_metric_distribution(
    comparison_effective_aggregate_metrics,
    metric_name="effective_RMSE",
    group_columns=[*comparison_base_model_columns, "split_name", "train_size"],
)
rmse_gap_table = (
    rmse_train_test_summary
    .pivot_table(
        index=["model_name", "model_label", "model_family", "model_order", "train_size"],
        columns="split_name",
        values="metric_mean",
        aggfunc="first",
    )
    .reset_index()
)
rmse_gap_table["Delta RMSE"] = rmse_gap_table["test"] - rmse_gap_table["train"]
rmse_gap_largest = (
    rmse_gap_table.loc[rmse_gap_table["train_size"] == comparison_largest_train_size]
    .loc[:, ["model_label", "Delta RMSE"]]
    .rename(columns={"model_label": "Model"})
)

scaling_table = final_rmse_table.merge(rmse_nauc_table, on="Model", how="left")
scaling_table = scaling_table.merge(sample_rank_table, on="Model", how="left")
scaling_table = scaling_table.merge(rmse_gap_largest, on="Model", how="left")
scaling_table["Model"] = pd.Categorical(scaling_table["Model"], categories=model_display_order, ordered=True)
scaling_table = scaling_table.sort_values("Model").reset_index(drop=True)
scaling_table["Model"] = scaling_table["Model"].astype(str)

violation_tolerance = 1e-10
physical_rows = []
for model_name, spec in retained_model_specs.items():
    report = fixed_split_results[model_name]["test_report"]
    fractional_key = "projected_fractional_predictions" if "projected_fractional_predictions" in report else "raw_fractional_predictions"
    constraint_key = "projected_constraint_l2" if "projected_constraint_l2" in report["constraint_residuals"].columns else "raw_constraint_l2"
    fractional_values = report[fractional_key].to_numpy(dtype=float)
    constraint_values = report["constraint_residuals"][constraint_key].to_numpy(dtype=float)
    physical_rows.append(
        {
            "Model": spec["label"],
            "Mass-Conservation Violations (%)": 100.0 * float(np.mean(constraint_values > violation_tolerance)),
            "Non-Negativity Violations (%)": 100.0 * float(np.mean((fractional_values < -violation_tolerance).any(axis=1))),
        }
    )
physical_table = pd.DataFrame(physical_rows)
physical_table["Model"] = pd.Categorical(physical_table["Model"], categories=model_display_order, ordered=True)
physical_table = physical_table.sort_values("Model").reset_index(drop=True)
physical_table["Model"] = physical_table["Model"].astype(str)

icsor_effective_coefficients = fixed_split_results["icsor"]["model_bundle"]["effective_coefficients"]
measured_targets = list(metadata["measured_output_columns"])
cod_target_index = measured_targets.index("COD")

def target_block_values(block_values: np.ndarray, target_index: int) -> np.ndarray:
    values_array = np.asarray(block_values, dtype=float)
    if values_array.ndim == 1:
        return values_array[target_index : target_index + 1]
    return values_array[target_index]

cod_block_arrays = {
    "b": np.asarray(target_block_values(icsor_effective_coefficients["b"], cod_target_index), dtype=float),
    "W_u": np.asarray(target_block_values(icsor_effective_coefficients["W_u"], cod_target_index), dtype=float),
    "W_in": np.asarray(target_block_values(icsor_effective_coefficients["W_in"], cod_target_index), dtype=float),
    "Theta_uu": np.asarray(target_block_values(icsor_effective_coefficients["Theta_uu"], cod_target_index), dtype=float),
    "Theta_uc": np.asarray(target_block_values(icsor_effective_coefficients["Theta_uc"], cod_target_index), dtype=float),
    "Theta_cc": np.asarray(target_block_values(icsor_effective_coefficients["Theta_cc"], cod_target_index), dtype=float),
}
all_cod_abs = np.concatenate([np.abs(values.ravel()) for values in cod_block_arrays.values()])
cod_retention_threshold = retention_fraction * float(all_cod_abs.max() if all_cod_abs.size else 0.0)

regularization_rows = []
for block_key, block_label in [
    ("b", "b"),
    ("W_u", "W_u"),
    ("W_in", "W_in"),
    ("Theta_uu", "Theta_uu"),
    ("Theta_uc", "Theta_uc"),
    ("Theta_cc", "Theta_cc"),
]:
    block_values = cod_block_arrays[block_key]
    total_count = int(block_values.size)
    retained_count = int(np.sum(np.abs(block_values) >= cod_retention_threshold)) if cod_retention_threshold > 0.0 else total_count
    regularization_rows.append(
        {
            "Coefficient block": block_label,
            "Total coeffs.": total_count,
            "Retained coeffs.": retained_count,
            "% Retained": (100.0 * retained_count / total_count) if total_count > 0 else 0.0,
        }
    )

regularization_table = pd.DataFrame(regularization_rows)
regularization_table = pd.concat(
    [
        regularization_table,
        pd.DataFrame(
            [
                {
                    "Coefficient block": "Overall summary",
                    "Total coeffs.": int(regularization_table["Total coeffs."].sum()),
                    "Retained coeffs.": int(regularization_table["Retained coeffs."].sum()),
                    "% Retained": 100.0
                    * float(regularization_table["Retained coeffs."].sum())
                    / float(max(1, regularization_table["Total coeffs."].sum())),
                }
            ]
        ),
    ],
    ignore_index=True,
)

theta_cc_cod = np.asarray(target_block_values(icsor_effective_coefficients["Theta_cc"], cod_target_index), dtype=float)

gamma_matrix = np.asarray(fixed_split_results["icsor"]["model_bundle"].get("Gamma_matrix", np.zeros((len(state_columns), len(state_columns)))), dtype=float)
if gamma_matrix.shape != (len(state_columns), len(state_columns)):
    gamma_matrix = np.zeros((len(state_columns), len(state_columns)), dtype=float)

def build_target_atlas_blocks(target_name: str) -> dict[str, np.ndarray]:
    target_index = measured_targets.index(target_name)
    return {
        "b": np.asarray([[float(target_block_values(icsor_effective_coefficients["b"], target_index).ravel()[0])]], dtype=float),
        "W_u": np.asarray(target_block_values(icsor_effective_coefficients["W_u"], target_index), dtype=float).reshape(1, -1),
        "W_in": np.asarray(target_block_values(icsor_effective_coefficients["W_in"], target_index), dtype=float).reshape(1, -1),
        "Theta_uu": np.asarray(target_block_values(icsor_effective_coefficients["Theta_uu"], target_index), dtype=float),
        "Theta_uc": np.asarray(target_block_values(icsor_effective_coefficients["Theta_uc"], target_index), dtype=float),
        "Theta_cc": np.asarray(target_block_values(icsor_effective_coefficients["Theta_cc"], target_index), dtype=float),
        "Gamma": gamma_matrix.copy(),
    }

print("Prepared all derived tables and figure tensors for export cells.")
print(f"Largest training size: {comparison_largest_train_size}")
print(f"COD retention threshold: {cod_retention_threshold:.6e}")

In [ ]:
def plot_target_atlas(target_name: str) -> plt.Figure:
    return plot_icsor_target_atlas(
        build_target_atlas_blocks(target_name),
        target_name=target_name,
        operational_labels=list(metadata["operational_columns"]),
        state_labels=state_columns,
        colorbar_label="Coefficient value",
        include_footer=False,
    )

In [ ]:
benchmark_table_source = benchmark_table.copy()
benchmark_metric_columns = ["RMSE", "MAE", "R2", "MAPE"]
if (
    not set(["Model", *benchmark_metric_columns]).issubset(benchmark_table_source.columns)
    or benchmark_table_source[benchmark_metric_columns].isna().all().all()
):
    benchmark_rows = []
    for model_name, spec in retained_model_specs.items():
        report_metrics = fixed_split_results[model_name]["test_report"]["aggregate_metrics"]
        preferred_type = "projected" if "projected" in set(report_metrics["prediction_type"]) else "raw"
        effective_row = report_metrics.loc[report_metrics["prediction_type"] == preferred_type].iloc[0]
        benchmark_rows.append({
            "Model": spec["label"],
            "RMSE": float(effective_row["RMSE"]),
            "MAE": float(effective_row["MAE"]),
            "R2": float(effective_row["R2"]),
            "MAPE": float(effective_row["MAPE"]),
        })
    benchmark_table_source = pd.DataFrame(benchmark_rows).sort_values("RMSE").reset_index(drop=True)

benchmark_table_rounded = benchmark_table_source.copy()
for metric_column in benchmark_metric_columns:
    benchmark_table_rounded[metric_column] = benchmark_table_rounded[metric_column].astype(float).round(4)

benchmark_export_path = export_table_csv("table_results_benchmark_sample", benchmark_table_rounded, index=False)
ipython_display(benchmark_table_rounded)

In [ ]:
target_table_rounded = target_table.copy()
target_rounding_digits: dict[str, int] = {}
for target_column in comparison_target_labels:
    raw_values = pd.to_numeric(target_table[target_column], errors="coerce")
    decimals = 4
    rounded_values = raw_values.round(decimals)
    while (
        raw_values.nunique(dropna=True) > rounded_values.nunique(dropna=True)
        and decimals < 8
    ):
        decimals += 1
        rounded_values = raw_values.round(decimals)
    target_rounding_digits[target_column] = decimals
    target_table_rounded[target_column] = rounded_values

adjusted_target_rounding = {
    target_column: decimals
    for target_column, decimals in target_rounding_digits.items()
    if decimals != 4
}
if adjusted_target_rounding:
    print("Increased target-table precision to avoid collapsing distinct values:")
    for target_column, decimals in adjusted_target_rounding.items():
        print(f"- {target_column}: {decimals} decimals")

if "TP" in adjusted_target_rounding:
    tp_raw_values = pd.to_numeric(target_table["TP"], errors="coerce")
    tp_display_values = pd.to_numeric(target_table_rounded["TP"], errors="coerce")
    print(f"TP distinct values preserved: {tp_display_values.nunique(dropna=True)} of {tp_raw_values.nunique(dropna=True)}")

target_export_path = export_table_csv("table_results_target_sample", target_table_rounded, index=False)
ipython_display(target_table_rounded)

In [ ]:
rmse_curve_plot_frame = rmse_learning_curve_summary.copy()
rmse_curve_plot_frame["model_label"] = pd.Categorical(
    rmse_curve_plot_frame["model_label"],
    categories=model_display_order,
    ordered=True,
)
rmse_curve_plot_frame = rmse_curve_plot_frame.sort_values(["model_label", "train_size"]).reset_index(drop=True)
rmse_curve_plot_frame["model_label"] = rmse_curve_plot_frame["model_label"].astype(str)

figure, axis = plot_metric_summary_lines(
    rmse_curve_plot_frame,
    x_column="train_size",
    y_column="metric_mean",
    group_column="model_label",
    lower_column="metric_q25",
    upper_column="metric_q75",
    title="Effective test RMSE across training sizes",
    x_label="Number of training samples",
    y_label="Effective test RMSE",
    legend_title="Model",
    figure_size=(10.2, 6.0),
    color_cycle=list(PIBRE_THEME_TOKENS["qualitative_cycle"]),
    marker_cycle=list(PIBRE_THEME_TOKENS["line_marker_cycle"]),
    linestyle_cycle=list(PIBRE_THEME_TOKENS["line_style_cycle"]),
    legend_outside=True,
    legend_location="bottom",
    legend_columns=5,
)

figure2_export_path = export_figure_pdf("figure2_rmse_learning_curve", figure)
ipython_display(figure)
plt.close(figure)

In [ ]:
scaling_table_rounded = scaling_table.copy()
for column_name in ["Final RMSE", "RMSE nAUC", "Sample Avg. Rank", "Delta RMSE"]:
    scaling_table_rounded[column_name] = scaling_table_rounded[column_name].astype(float).round(4)

scaling_export_path = export_table_csv("table_results_scaling_sample", scaling_table_rounded, index=False)
ipython_display(scaling_table_rounded)

In [ ]:
runtime_rows = []
for model_name, spec in retained_model_specs.items():
    model_runtime = analysis_results_by_model[model_name]["run_metadata"].copy()
    runtime_rows.append(
        model_runtime.assign(
            model_name=model_name,
            model_label=spec["label"],
            model_family=spec["family"],
        )
    )

runtime_frame = pd.concat(runtime_rows, ignore_index=True)
runtime_summary = summarize_metric_distribution(
    runtime_frame,
    metric_name="elapsed_seconds",
    group_columns=["model_name", "model_label", "model_family", "train_size"],
)
runtime_summary["model_label"] = pd.Categorical(
    runtime_summary["model_label"],
    categories=model_display_order,
    ordered=True,
)
runtime_summary = runtime_summary.sort_values(["model_label", "train_size"]).reset_index(drop=True)
runtime_summary["model_label"] = runtime_summary["model_label"].astype(str)

figure, axis = plot_metric_summary_lines(
    runtime_summary,
    x_column="train_size",
    y_column="metric_mean",
    group_column="model_label",
    lower_column="metric_q25",
    upper_column="metric_q75",
    title="Training runtime across training sizes",
    x_label="Number of training samples",
    y_label="Training runtime (s)",
    legend_title="Model",
    figure_size=(10.2, 6.0),
    color_cycle=list(PIBRE_THEME_TOKENS["qualitative_cycle"]),
    marker_cycle=list(PIBRE_THEME_TOKENS["line_marker_cycle"]),
    linestyle_cycle=list(PIBRE_THEME_TOKENS["line_style_cycle"]),
    legend_outside=True,
    legend_location="bottom",
    legend_columns=5,
)
axis.set_yscale("log")
axis.grid(True, which="both", linestyle=":", alpha=0.35)

figure3_export_path = export_figure_pdf("figure3_runtime_learning_curve", figure)
ipython_display(figure)
plt.close(figure)

In [ ]:
violation_tolerance = 1e-10
physical_rows = []
for model_name, spec in retained_model_specs.items():
    report = fixed_split_results[model_name]["test_report"]
    final_prediction_type = "projected" if model_name == "icsor" else "raw"
    fractional_key = f"{final_prediction_type}_fractional_predictions"
    if fractional_key not in report:
        fractional_key = "raw_fractional_predictions" if "raw_fractional_predictions" in report else "projected_fractional_predictions"
    constraint_key = f"{final_prediction_type}_constraint_l2"
    constraint_frame = report["constraint_residuals"]
    if constraint_key not in constraint_frame.columns:
        constraint_key = "raw_constraint_l2" if "raw_constraint_l2" in constraint_frame.columns else "projected_constraint_l2"
    fractional_values = report[fractional_key].to_numpy(dtype=float)
    constraint_values = constraint_frame[constraint_key].to_numpy(dtype=float)
    physical_rows.append(
        {
            "Model": spec["label"],
            "Mass-Conservation Violations (%)": 100.0 * float(np.mean(constraint_values > violation_tolerance)),
            "Non-Negativity Violations (%)": 100.0 * float(np.mean((fractional_values < -violation_tolerance).any(axis=1))),
        }
    )

physical_table = pd.DataFrame(physical_rows)
physical_table["Model"] = pd.Categorical(physical_table["Model"], categories=model_display_order, ordered=True)
physical_table = physical_table.sort_values("Model").reset_index(drop=True)
physical_table["Model"] = physical_table["Model"].astype(str)

physical_table_rounded = physical_table.copy()
for metric_column in ["Mass-Conservation Violations (%)", "Non-Negativity Violations (%)"]:
    physical_table_rounded[metric_column] = physical_table_rounded[metric_column].astype(float).round(4)

physical_export_path = export_table_csv("table_results_physical_sample", physical_table_rounded, index=False)
ipython_display(physical_table_rounded)

In [ ]:
figure, axis = plot_coefficient_heatmap(
    theta_cc_cod,
    row_labels=state_columns,
    column_labels=state_columns,
    title=r"COD-specific $\Theta_{cc}$ heatmap",
    x_label="Influent ASM component",
    y_label="Influent ASM component",
    colorbar_label="Coefficient value",
    figure_size=(9.6, 7.8),
    x_tick_rotation=62.0,
)

figure4_export_path = export_figure_pdf("figure4_icsor_structure", figure)
ipython_display(figure)
plt.close(figure)

In [ ]:
# Rebuild the export table from live COD-specific blocks so Theta_uc and Gamma are never dropped.
def _select_cod_block(block_name: str) -> np.ndarray:
    block_values = np.asarray(icsor_effective_coefficients[block_name], dtype=float)
    if block_values.ndim == 1:
        return block_values[cod_target_index : cod_target_index + 1]
    return block_values[cod_target_index]


def _summarize_regularization_block(
    block_label: str,
    block_values: np.ndarray,
    *,
    exclude_diagonal: bool = False,
) -> dict[str, object]:
    block_array = np.asarray(block_values, dtype=float)
    selectable_mask = np.ones(block_array.shape, dtype=bool)
    if exclude_diagonal and block_array.ndim == 2 and block_array.shape[0] == block_array.shape[1]:
        np.fill_diagonal(selectable_mask, False)

    total_count = int(selectable_mask.sum())
    selectable_values = np.abs(block_array[selectable_mask]) if total_count > 0 else np.asarray([], dtype=float)
    if total_count == 0:
        retained_count = 0
    elif cod_retention_threshold > 0.0:
        retained_count = int(np.sum(selectable_values >= cod_retention_threshold))
    else:
        retained_count = total_count

    return {
        "Coefficient block": block_label,
        "Total coeffs.": total_count,
        "Retained coeffs.": retained_count,
        "% Retained": 0.0 if total_count == 0 else 100.0 * retained_count / total_count,
    }


regularization_block_specs = [
    ("b", _select_cod_block("b"), False),
    ("W_u", _select_cod_block("W_u"), False),
    ("W_in", _select_cod_block("W_in"), False),
    ("Theta_uu", _select_cod_block("Theta_uu"), False),
    ("Theta_uc", _select_cod_block("Theta_uc"), False),
    ("Theta_cc", _select_cod_block("Theta_cc"), False),
    ("Gamma", np.asarray(gamma_matrix, dtype=float), True),
]

regularization_table_source = pd.DataFrame(
    [
        _summarize_regularization_block(
            block_label,
            block_values,
            exclude_diagonal=exclude_diagonal,
        )
        for block_label, block_values, exclude_diagonal in regularization_block_specs
    ]
)
regularization_table = pd.concat(
    [
        regularization_table_source,
        pd.DataFrame(
            [
                {
                    "Coefficient block": "Overall summary",
                    "Total coeffs.": int(regularization_table_source["Total coeffs."].sum()),
                    "Retained coeffs.": int(regularization_table_source["Retained coeffs."].sum()),
                    "% Retained": 100.0
                    * float(regularization_table_source["Retained coeffs."].sum())
                    / float(max(1, regularization_table_source["Total coeffs."].sum())),
                }
            ]
        ),
    ],
    ignore_index=True,
)

expected_regularization_blocks = {
    "b",
    "W_u",
    "W_in",
    "Theta_uu",
    "Theta_uc",
    "Theta_cc",
    "Gamma",
    "Overall summary",
}
missing_regularization_blocks = expected_regularization_blocks.difference(
    set(regularization_table["Coefficient block"])
 )
if missing_regularization_blocks:
    raise RuntimeError(
        "Regularization export is missing required coefficient blocks: "
        f"{sorted(missing_regularization_blocks)}"
    )

regularization_table_rounded = regularization_table.copy()
regularization_table_rounded["Total coeffs."] = regularization_table_rounded["Total coeffs."].astype(int)
regularization_table_rounded["Retained coeffs."] = regularization_table_rounded["Retained coeffs."].astype(int)
regularization_table_rounded["% Retained"] = regularization_table_rounded["% Retained"].astype(float).round(4)

regularization_export_path = export_table_csv("table_results_icsor_regularization_sample", regularization_table_rounded, index=False)
ipython_display(regularization_table_rounded)

In [ ]:
figure = plot_target_atlas("COD")
figureS1_export_path = export_figure_pdf("figureS1_cod_icsor_structure", figure)
ipython_display(figure)
plt.close(figure)

In [ ]:
figure = plot_target_atlas("TN")
figureS2_export_path = export_figure_pdf("figureS2_tn_icsor_structure", figure)
ipython_display(figure)
plt.close(figure)

In [ ]:
figure = plot_target_atlas("TP")
figureS3_export_path = export_figure_pdf("figureS3_tp_icsor_structure", figure)
ipython_display(figure)
plt.close(figure)

In [ ]:
figure = plot_target_atlas("TSS")
figureS4_export_path = export_figure_pdf("figureS4_tss_icsor_structure", figure)
ipython_display(figure)
plt.close(figure)

In [ ]:
expected_asset_paths = [submission_asset_dir / asset_name for asset_name in required_assets]
asset_status = pd.DataFrame(
    {
        "asset": [path.name for path in expected_asset_paths],
        "exists": [path.exists() for path in expected_asset_paths],
        "path": [str(path) for path in expected_asset_paths],
    }
)

print("Comparison notebook data provenance:")
print(f"- Dataset CSV: {artifact_paths['dataset_csv']}")
print(f"- Metadata JSON: {artifact_paths['metadata_json']}")
print(f"- Matrix source: {matrix_bundle['composition_workbook_path']}")
print(f"- Composition cache source: {matrix_bundle['composition_cache_source']}")

if any("/results/" in str(path).replace("\\", "/") for path in expected_asset_paths):
    raise RuntimeError("One or more export targets point to the results directory, which is not allowed.")

print("Asset existence check:")
ipython_display(asset_status)
if not bool(asset_status["exists"].all()):
    missing_assets = asset_status.loc[~asset_status["exists"], "asset"].tolist()
    raise FileNotFoundError(f"Missing exported assets: {missing_assets}")

print("All expected manuscript and supplementary assets are present in docs/DCHE-D-26-00020.")